# Dual-Pathway Pruning (Using Vertex AI and DuckDB)


## Install piglets


If running from within the piglets repository:
```bash
    uv sync --extra examples --extra google_vertexai --extra duckdb
``` 
followed by the import below:


In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


If running from outside the piglets repository:
```bash
    uv add piglets[google_vertexai,duckdb,examples]
``` 


## Set Your Model, Database and Natural Language Question

In [ ]:
MODEL_NAME = "gemini-2.5-flash"
DATABASE_TYPE = "duckdb"
DB_PATH = "data/tpch_sf1.duckdb"
QUESTION =  """
    Which manufacturers saw the largest increase in average revenue per order between 1996 and 1997, 
    considering only manufacturers with at least 100 orders in both years, and excluding cancelled orders?
"""

## Create the DuckDB Database `tpch_sf1` Locally

In [ ]:
from scripts.create_example_db import create_tpch_example_duckdb_db

create_tpch_example_duckdb_db(db_path=DB_PATH)

## Step 1: Create a Logical Plan


We first have our model verbalise it's hypotheses. This step is schema agnostic and is therefore completed without any knowledge of the actual database. This allows the model to reason about possible approaches without the influence of naming conventions and ambiguous abbreviations in the database schema.

The `piglets` logical planner creates `N` logical plans before aggregating them into a single plan. This is to ensure less potential hypotheses are missed. The approach here errs on the side of maximising recall, therefore we would rather have too many hypotheses to validate than include one that is valuable. In the `LogicalPlanner` the number of logcial plans we create before aggregation is determined by `num_samples`, in this case we have a sample size of 3.


In [ ]:
from piglets import LogicalPlanner


logical_planner = LogicalPlanner(MODEL_NAME)
logical_plan = logical_planner.plan(
    natural_language_query=QUESTION,
    num_samples=3,
)


Below we can inspect the sample plans that were first created and the aggregate plan that was created form them:


In [ ]:
print("------------------------------------")
print("Natural Language Question:")
print(logical_plan.natural_language_query)
print("------------------------------------")
print("Sample Plan 1:")
for i, step in enumerate(logical_plan.sample_plans[0].logical_steps, 1):
    print(f"Step {step}")
print("------------------------------------")
print("Sample Plan 2:")
for i, step in enumerate(logical_plan.sample_plans[1].logical_steps, 1):
    print(f"Step {step}")
print("------------------------------------")
print("Sample Plan 3:")
for i, step in enumerate(logical_plan.sample_plans[2].logical_steps, 1):
    print(f"Step {step}")
print("------------------------------------")
print("\nAggregated Logical Plan:")
for i, step in enumerate(logical_plan.logical_steps, 1):
    print(f"Step {step}")
print("------------------------------------")


## Step 2: Conenct to DuckDB and retrieve the Database Schema


Next we connect to DuckDB. The dataset `tpch_sf1` was created by running `create_tpch_example_duckdb_db` above.
Once connected we can retrieve information about our database which be useful context for future LLM calls.


In [ ]:
from piglets import DuckDBURL, DatabaseConnector

database_connector = DatabaseConnector(
    connection=DuckDBURL(
        database=DB_PATH,
    ),
)
database = database_connector.get_database_schema()


In [ ]:
print("------------------------------------")
print("Database Name: ")
print(database.name)
print("------------------------------------")
print("Database Tables: ")
for table in database.tables:
    print("------------------------------------")
    print(f"Table: {table.name}")
    print("Columns:")
    for column in table.columns:
        print(f"  - {column.name} ({column.data_type})")
print("------------------------------------")


## Use Dual-Pathway Pruning to Reduce Potemtial Schema


Above we have retrieved a large schema, much of which won't be relevant to our natural language question.
The `piglets` `Pruner` uses dual-pathway pruning to reduce the size of this schema.
Dual-pathway pruning first produces a `PreservationSet` and a `DeletionSet`, these are made up of potentially useful tables and columns and obviously useless tables and columns respectively. 
We then take the union of all columns either not present in the `DeletionSet` or are present in the `PreservationSet`. More formally:
$$
D_{\text{pruned}} = (D \setminus C_{\text{del}}) \cup C_{\text{keep}}
$$
for the database schema $D$, a preservation set $C_{\text{keep}}$ and a deletion set $C_{\text{del}}$.


In [ ]:
from piglets import Pruner

pruner = Pruner(model_name=MODEL_NAME)
pruned_database = pruner.dual_pathway_pruning(
    natural_language_query=QUESTION,
    database=database,
    logical_plan=logical_plan,
)


In [ ]:
print("------------------------------------")
print("Database Name: ")
print(pruned_database.name)
print("------------------------------------")
print("Database Tables: ")
for table in pruned_database.tables:
    print("------------------------------------")
    print(f"Table: {table.name}")
    print("Columns:")
    for column in table.columns:
        print(f"  - {column.name} ({column.data_type})")
print("------------------------------------")


You should now see a reduced version of the original schema above.
